In [ ]:
import os
import numpy as np
import pandas as pd
import squidpy as sq
import scanpy as sc
import liana as li
import cell2cell as c2c
import scvi
import decoupler as dc
import matplotlib.pyplot as plt
import seaborn as sns
from aquarel import load_theme
import cmcrameri

from pathlib import Path
from matplotlib.colors import ListedColormap
from scipy.sparse import issparse, csr_matrix
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean

In [ ]:
WORKING_DIR = "../differential_expression"
DATA_DIR = "../cell-type_deconvolution/data"
ADATA = f"{DATA_DIR}/adata.h5ad"
OUT_DIR = os.path.join(WORKING_DIR, "figures")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

sc.settings.figdir = OUT_DIR
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

theme = (
    load_theme("umbra_light").set_overrides({
        "figure.facecolor": 'white',
        "axes.facecolor": 'white'
    })
    .set_grid(draw=False)
)
theme.apply()
adata = sc.read(ADATA)
adata

In [ ]:
adata.obs[adata.uns['mod']['factor_names']] = adata.obsm['q05_cell_abundance_w_sf']
adata.obs

In [ ]:
from cell2location import run_colocation

adata = adata[~adata.obs['class'].isin(['Marker'])].copy()

res_dict, adata_vis = run_colocation(
    adata,
    model_name='CoLocatedGroupsSklearnNMF',
    train_args={
      'n_fact': np.arange(10, 15), # IMPORTANT: use a wider range of the number of factors (5-30)
      'sample_name_col': 'sample', # columns in adata_vis.obs that identifies sample
      'n_restarts': 3 # number of training restarts
    },
    # the hyperparameters of NMF can be also adjusted:
    model_kwargs={'alpha': 0.01, 'init': 'random', "nmf_kwd_args": {"tol": 0.000001}},
    export_args={'path': f'{OUT_DIR}/CoLocatedComb/'}
)

In [ ]:
n_fact = 10
mod_facts = adata_vis.uns[f"mod_coloc_n_fact{n_fact}"]
W = mod_facts['post_sample_means']['location_factors']
factor_names = mod_facts['fact_names']

# Add each factor to obs
for i, fname in enumerate(factor_names):
    adata_vis.obs[fname] = W[:, i]

sc.pl.spatial(adata_vis[adata_vis.obs['sample'] == 'Paxgene3'].copy(),
              color=factor_names,
              library_id='Paxgene3',
              cmap='magma',
              size=1.3,
              ncols=4,
              img_key=None,
              title=[f"Zone {i}" for i in range(len(factor_names))])

In [ ]:
factor_names

In [ ]:
Q = mod_facts['post_sample_means']['cell_type_factors']
cell_types = mod_facts['var_names']

df_q = pd.DataFrame(Q, index=cell_types, columns=factor_names)
subset_zones = ['fact_0', 'fact_1', 'fact_2', 'fact_3', 'fact_4', 'fact_5', 'fact_6', 'fact_7', 'fact_8', 'fact_9']
subset_zones_names = ['Cancer', 'Fibroblast', 'Epithelial', 'fact3', 'Mixed', 'Myeloid', 'fact6', 'B cell', 'fact8', 'fact9']
df_q = df_q[subset_zones]
df_q.columns = subset_zones_names

with theme:
    df_q.T.plot(kind='bar', stacked=True, figsize=(6, 4), cmap='cmc.lipari')
    plt.ylabel("Cell type contribution", ha='right')
    plt.xticks(rotation=45)
    plt.title("Composition of Colocalization Compartments")
    # plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
Q = mod_facts['post_sample_means']['cell_type_factors']
cell_types = mod_facts['var_names']

df_q = pd.DataFrame(Q, index=cell_types, columns=factor_names)
subset_zones = ['fact_0', 'fact_1', 'fact_2', 'fact_4', 'fact_5', 'fact_7', 'fact_9']
subset_zones_names = ['Cancer', 'Fibroblast', 'Epithelial', 'Mixed', 'Myeloid', 'B cell', 'T cell']
df_q = df_q[subset_zones]
df_q.columns = subset_zones_names

with theme:
    df_q.T.plot(kind='bar', stacked=True, figsize=(6, 4), cmap='cmc.lipari')
    plt.ylabel("Cell type contribution", ha='right')
    plt.xticks(rotation=45)
    plt.title("Composition of Colocalization Compartments")
    # plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig(OUT_DIR + '/compartments.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
g = sns.clustermap(df_q, cmap='cmc.lipari', figsize=(5, 5))

g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, ha='left')
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
g.ax_heatmap.set_xlabel("Compartment")
g.ax_heatmap.set_ylabel("Cell type")
plt.title("Cell Type Colocalization Across Compartments", fontsize=16)
plt.savefig(f"{OUT_DIR}/visualizations/heatmap_compartments.png", bbox_inches='tight', dpi=600)
plt.show()

In [ ]:
for k, v in mod_facts['post_sample_means'].items():
    try:
        print(k, np.shape(v))
    except Exception:
        print(k, type(v))

In [ ]:
W = mod_facts['post_sample_means']['location_factors']

df_w = pd.DataFrame(
    W,
    index=adata_vis.obs_names,
    columns=factor_names
)

df_w = df_w[subset_zones]
df_w.columns = subset_zones_names

adata_vis.obs["major_cell_type"] = pd.Categorical(
    df_w.idxmax(axis=1),
    categories=subset_zones_names
)

In [ ]:
adata = adata_vis.copy()
adata.obs["major_cell_type"]

In [ ]:
for slide in adata_vis.obs['sample'].unique():
    sq.pl.spatial_scatter(adata_vis[adata_vis.obs['sample'] == slide].copy(),
                  color='major_cell_type',
                  title=slide,
                  library_id=slide,
                  cmap='magma',
                  size=1.4,
                  img=None)
    plt.savefig(OUT_DIR + f'/{slide}_niches.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(adata.obs['total_counts'], kde=True, color='gray', label='Density')
plt.axvline(x=700, color='red', linestyle='--', label='Threshold (700)')
plt.title('Distribution of Library Sizes')
plt.xlabel('Library Size (Total Counts)')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith("MT-")

# Recalculate mitochondrial QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],    # Use the updated 'mt' column
    percent_top=None,  # Skip top expressed genes for now
    inplace=True       # Add results directly to adata.obs
)
sc.pl.highest_expr_genes(adata, n_top=15)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Total counts
sns.histplot(adata.obs["total_counts"], kde=False, bins=50, ax=axs[0])
axs[0].set_title("Total Counts")
axs[0].set_xlabel("Counts")
axs[0].set_ylabel("Frequency")

# Number of genes by counts
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=50, ax=axs[1])
axs[1].set_title("Number of Genes by Counts")
axs[1].set_xlabel("Number of Genes")
axs[1].set_ylabel("Frequency")

# Mitochondrial content
sns.histplot(adata.obs["pct_counts_mt"], kde=False, bins=50, ax=axs[2])
axs[2].set_title("Mitochondrial Content")
axs[2].set_xlabel("Mitochondrial %")
axs[2].set_ylabel("Frequency")

# Adjust layout for better visualization
plt.tight_layout()
plt.show()

In [ ]:
sc.pp.filter_cells(adata, min_counts=250, inplace=True)
sc.pp.filter_cells(adata, min_genes=50, inplace=True)
sc.pp.filter_genes(adata, min_cells=3, inplace=True)

In [ ]:
adata.layers['counts'] = adata.X.copy()

raw_counts_layer = 'counts' # raw UMI counts layer
batch_col = 'patient' # metadata colum specifying batches
scvi.model.SCVI.setup_anndata(adata, layer=raw_counts_layer, batch_key=batch_col)
model = scvi.model.SCVI(adata, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
model.train()
corrected_data = model.get_normalized_expression(transform_batch=sorted(adata.obs[batch_col].unique()),
                                                 library_size=1e4)
corrected_data.iloc[:,:] = np.log1p(corrected_data.values)
adata.X = corrected_data

In [ ]:
original_counts = adata.layers['counts'].sum(axis=1)
normalized_counts = adata.X.sum(axis=1)

plt.figure(figsize=(10, 5))
sns.histplot(original_counts.A1, color="blue", label="Before Normalization", kde=True) 
sns.histplot(normalized_counts, color="orange", label="After Normalization", kde=True)
plt.xlabel("Total Expression per Cell")
plt.ylabel("Number of Cells")
plt.title("Effect of Log-Normalization on Expression Distribution")
plt.legend()
plt.show()

In [ ]:
adata.obs['sample_context'] = adata.obs["class"].astype(str) + "_" + adata.obs["PFI"].astype(str) + "_" + adata.obs["patient"].astype(str)

In [ ]:
adata.obs['sample_context'].value_counts()

In [ ]:
liana_res_raw = li.mt.rank_aggregate.by_sample(adata,
                                         sample_key='sample_context',
                                         groupby='major_cell_type',
                                         resource_name='consensus',
                                         expr_prop=0.9, # must be expressed in expr_prop fraction of cells
                                         min_cells=10,
                                         n_perms=10,
                                         use_raw=False, # run on batch corrected, and log- and library-normalized counts
                                         verbose=True,
                                         inplace=False,
                                         return_all_lrs=True
                                         )

In [ ]:
liana_res = liana_res_raw.copy() #[
#     (liana_res_raw["magnitude_rank"] >= 0.05) &
#     (liana_res_raw["specificity_rank"] <= 0.01)
# ].copy()
# liana_res.sort_values(by="magnitude_rank", ascending=False)

In [ ]:
li.method.show_methods()

In [ ]:
# convert to long format by index, and each score and value in different columns
liana_res_long = liana_res.loc[:, liana_res.columns[~liana_res.columns.str.contains(pat = 'rank')]]
liana_res_long = liana_res_long.melt(id_vars=['source', 'target', 'ligand_complex', 'receptor_complex'], var_name='score', value_name='value')

liana_res_long['score'] = liana_res_long['score'].astype('category')

In [ ]:
import matplotlib

magnitude_scores = ['lr_means', 'expr_prod', 'lrscore', 'lr_probs']
avail_scores = [c for c in magnitude_scores if c in liana_res.columns]

liana_corr = liana_res[avail_scores].corr(method='spearman')
print(liana_corr)

cm = c2c.plotting.clustermap_cci(liana_corr,
                                 method='ward',
                                 optimal_leaf=True,
                                 metadata=None,
                                 title='',
                                 cbar_title='Similarity',
                                 cmap='Blues_r',
                                 vmax=1.,
#                                  vmin=0.,
                                 annot=True,
                                 dendrogram_ratio=0.15,
                                figsize=(4,5))

font = matplotlib.font_manager.FontProperties(weight='bold', size=7)
for ax in [cm.ax_heatmap, cm.ax_cbar]:
    for tick in ax.get_xticklabels():
        tick.set_fontproperties(font)
    for tick in ax.get_yticklabels():
        tick.set_fontproperties(font)

    text = ax.yaxis.label
    text.set_font_properties(font)

plt.show()

In [ ]:
adata.write_h5ad(os.path.join(OUT_DIR, 'processed.h5ad'), compression='gzip')
liana_res.to_csv(OUT_DIR + '/LIANA_by_sample.csv', index=False)

In [ ]:
import plotnine as p9

plot = (li.pl.dotplot_by_sample(liana_res=liana_res,
                               colour='magnitude_rank',
                               size='specificity_rank',
                               source_labels=["Cancer", "Epithelial", "Fibroblast", "Myeloid", "B cell"],
                               target_labels=["Cancer", "Epithelial", "Fibroblast", "Myeloid", "B cell"],
                               ligand_complex=['COL1A1', 'LAMA5'],
                               receptor_complex=['DDR1', 'BCAM'],
                               sample_key='sample_context',
                               inverse_colour=True,
                               inverse_size=True,
                               figure_size=(16, 9),
                               size_range=(1, 6),
                               ) +
         p9.scale_color_continuous(name="Magnitude rank") +
         p9.labs(size='Specificity Rank')
       )

plot.save(OUT_DIR + '/dotplot-by-sample.pdf', height=9, width=16)
plot

In [ ]:
pivot = liana_res.pivot_table(values='lr_means', index='source', columns='target', aggfunc='mean')
sns.heatmap(pivot, cmap='viridis')
plt.show()

In [ ]:
tensor = li.multi.to_tensor_c2c(liana_res=liana_res, # LIANA's dataframe containing results
                                sample_key='sample_context', # Column name of the samples
                                source_key='source', # Column name of the sender cells
                                target_key='target', # Column name of the receiver cells
                                ligand_key='ligand_complex', # Column name of the ligands
                                receptor_key='receptor_complex', # Column name of the receptors
                                score_key='magnitude_rank', # Column name of the communication scores to use
                                non_negative=True, # set negative values to 0
                                inverse_fun=lambda x: 1 - x, # Transformation function
                                non_expressed_fill=None, # Value to replace missing values with
                                how='outer', # What to include across all samples
                                outer_fraction=1/3., # Fraction of samples as threshold to include cells and LR pairs.
                                lr_fill=np.nan, # What to fill missing LRs with
                                cell_fill=np.nan, # What to fill missing cell types with
                                lr_sep='^', # How to separate ligand and receptor names to name LR pair
                                context_order=sorted(liana_res['sample_context'].unique()), # Order to store the contexts in the tensor
                                sort_elements=True # Whether sorting alphabetically element names of each tensor dim. Does not apply for context order if context_order is passed.
                               )

In [ ]:
from collections import defaultdict

element_dict = defaultdict(lambda: 'Unknown')
context_dict = element_dict.copy()

for phenotype, df in adata.obs.groupby('PFI'):
    for layer_context in df['sample_context'].unique():
        context_dict[layer_context] = phenotype

context_dict

In [ ]:
dimensions_dict = [context_dict, None, None, None]

In [ ]:
meta_tensor = c2c.tensor.generate_tensor_metadata(interaction_tensor=tensor,
                                                  metadata_dicts=dimensions_dict,
                                                  fill_with_order_elements=True
                                                  )

In [ ]:
for r in [2, 3, 4, 5, 6, 8, 10, 12, 14]:
    os.makedirs(f"{OUT_DIR}/rank_{r}", exist_ok=True)
    tensor_factorised = c2c.analysis.run_tensor_cell2cell_pipeline(
        tensor,
        meta_tensor,
        copy_tensor=True,
        random_state=0,
        rank=r,
        tf_init='svd',
        device='cpu',
        fig_fontsize=14,
        output_fig=True,
        output_folder=f"{OUT_DIR}/rank_{r}"
    )

    c2c.plotting.context_boxplot(context_loadings=tensor_factorised.factors["Contexts"],
                                     metadict=context_dict,
                                     nrows=2,
                                     figsize=(8, 6),
                                     statistical_test='t-test_ind',
                                     pval_correction='fdr_bh',
                                     cmap='cmc.lipari',
                                     verbose=False,
                                    )
    plt.savefig(OUT_DIR + f'/rank_{r}/context_boxplots_rank_{r}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
r = 12
tensor_factorised = c2c.analysis.run_tensor_cell2cell_pipeline(
    tensor,
    meta_tensor,
    copy_tensor=True,
    random_state=0,
    rank=r,
    tf_init='svd',
    device='cpu',
    fig_fontsize=14,
    output_fig=True,
    output_folder=f"{OUT_DIR}/rank_{r}"
)

In [ ]:
c2c.io.export_variable_with_pickle(tensor, OUT_DIR + '/tensor_factorized.pkl')

In [ ]:
c2c.io.export_variable_with_pickle(meta_tensor, OUT_DIR + '/tensor_metadata.pkl')

In [ ]:
with theme:
    fig, axes = c2c.plotting.tensor_factors_plot(
        interaction_tensor=tensor_factorised,
        metadata=meta_tensor,
        sample_col="Element",
        group_col="Category",
        fontsize=14,
        meta_cmaps = ['plasma', 'Dark2_r', 'tab20', 'tab20'],
    )
    
    fig.patch.set_facecolor("white")
    for ax in fig.axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    
    fig.savefig(
        "tensor_factors.pdf",
        bbox_inches="tight"
    )

In [ ]:
c2c.plotting.context_boxplot(context_loadings=tensor_factorised.factors["Contexts"],
                                 metadict=context_dict,
                                 nrows=2,
                                 figsize=(8, 6),
                                 statistical_test='t-test_ind',
                                 pval_correction='fdr_bh',
                                 cmap='cmc.lipari',
                                 verbose=False,
                                )
plt.savefig(OUT_DIR + '/context_boxplots.pdf', bbox_inches='tight')
plt.show()

In [ ]:
def severity_rank(x):
    if x == 'long':
        ranking = 1
    elif x == 'medium':
        ranking = 2
    elif x == 'short':
        ranking = 3
    return ranking

In [ ]:
sev_rank = pd.DataFrame(index=tensor_factorised.factors['Contexts'].index)
sev_rank['PFI'] = [severity_rank(context_dict[idx]) for idx in tensor_factorised.factors['Contexts'].index]
sev_rank.head()

In [ ]:
import scipy

nfactors = tensor_factorised.factors['Contexts'].shape[1]
for i in range(1, nfactors+1):
    factor = 'Factor {}'.format(i)
    print(factor, scipy.stats.spearmanr(tensor_factorised.factors['Contexts'][factor],
                                sev_rank['PFI']))

Factor 1 is associated with short outcome, whereas Factor 7 is associated with long survival.

In [ ]:
condition_colors = c2c.plotting.aesthetics.get_colors_from_labels(['long', 'medium', 'short'],
                                                                  cmap='plasma')

# Map these colors to each sample name
color_dict = {k : condition_colors[v] for k, v in context_dict.items()}

# Generate a dataframe used as input for the clustermap
col_colors = pd.Series(color_dict)
col_colors = col_colors.to_frame()
col_colors.columns = ['PFI']

In [ ]:
sample_cm = c2c.plotting.loading_clustermap(tensor_factorised.factors['Contexts'],
                                            use_zscore=True, # Whether standardizing the loadings across factors
                                            col_colors=col_colors, # Change this to color by other properties
                                            figsize=(16, 6),
                                            dendrogram_ratio=0.3,
                                            cbar_fontsize=12,
                                            tick_fontsize=14,
                                            filename=OUT_DIR + '/Clustermap-Contexts.pdf'
                                           )

plt.sca(sample_cm.ax_heatmap)
legend = c2c.plotting.aesthetics.generate_legend(color_dict=condition_colors,
                                                 bbox_to_anchor=(1.1, 0.5), # Position of the legend (X, Y)
                                                 title='PFI'
                                                )
plt.savefig(OUT_DIR + "/heatmap_context_loadings.pdf", bbox_inches='tight')
plt.show()

In [ ]:
lr_loadings = tensor_factorised.factors['Ligand-Receptor Pairs']
lr_loadings.sort_values("Factor 8", ascending=False).head(10)

In [ ]:
lr_loadings.sort_values("Factor 5", ascending=False).head(10)

In [ ]:
ccc_threshold = 0.15
c2c.plotting.ccc_networks_plot(tensor_factorised.factors,
                               included_factors=['Factor 1', 'Factor 7'],
                               ccc_threshold=ccc_threshold, # Only important communication
                               nrows=1,
                               panel_size=(16, 16), # This changes the size of each figure panel.
                              )
plt.savefig(OUT_DIR + "/comm_networks_pfi.pdf", bbox_inches='tight')
plt.show()

In [ ]:
networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(
    tensor_factorised.factors,
    sender_label="Sender Cells",
    receiver_label="Receiver Cells",
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pycirclize import Circos

factor = "Factor 1"
mat = networks[factor].copy()

# optional: keep only strong edges
ccc_threshold = 0.15
mat = mat.where(mat >= ccc_threshold, 0)

# remove empty rows/columns
mat = mat.loc[(mat.sum(axis=1) > 0), (mat.sum(axis=0) > 0)]

# make square matrix over the union of all cell types
nodes = sorted(set(mat.index) | set(mat.columns))
square = pd.DataFrame(0.0, index=nodes, columns=nodes)
square.loc[mat.index, mat.columns] = mat.values

# optional: reorder nodes manually for nicer biology-driven layout
preferred_order = [n for n in [
    "Cancer", "Epithelial", "Fibroblast", "Myeloid", "B cell", "T cell"
] if n in square.index]
remaining = [n for n in square.index if n not in preferred_order]
square = square.loc[preferred_order + remaining, preferred_order + remaining]

# optional: custom node colors
node_colors = {
    "Cancer": "#C44E52",
    "Epithelial": "#DD8452",
    "Fibroblast": "#55A868",
    "Myeloid": "#8172B3",
    "B cell": "#937860",
    "T cell": "#DA8BC3",
}

cmap = {n: node_colors.get(n, "#999999") for n in square.index}

circos = Circos.chord_diagram(
    square,
    space=4,
    cmap={n: node_colors.get(n, "#999999") for n in square.index},
    link_cmap=[(src, dst, node_colors.get(src, "#999999")) for src in square.index for dst in square.columns],
    label_kws=dict(size=12),
    link_kws=dict(alpha=0.65, lw=0.6, ec="white", direction=1, arrow_length_ratio=0.06),
)

fig = circos.plotfig(figsize=(4,4))
fig.suptitle(f"{factor} cell-cell communication program", y=1.02, fontsize=14)
fig.savefig("PFI_factor1_chord.pdf", bbox_inches="tight")
plt.show()

In [ ]:
factor = "Factor 7"
mat = networks[factor].copy()

# optional: keep only strong edges
ccc_threshold = 0.15
mat = mat.where(mat >= ccc_threshold, 0)

# remove empty rows/columns
mat = mat.loc[(mat.sum(axis=1) > 0), (mat.sum(axis=0) > 0)]

# make square matrix over the union of all cell types
nodes = sorted(set(mat.index) | set(mat.columns))
square = pd.DataFrame(0.0, index=nodes, columns=nodes)
square.loc[mat.index, mat.columns] = mat.values

# optional: reorder nodes manually for nicer biology-driven layout
preferred_order = [n for n in [
    "Cancer", "Epithelial", "Fibroblast", "Myeloid", "B cell", "T cell"
] if n in square.index]
remaining = [n for n in square.index if n not in preferred_order]
square = square.loc[preferred_order + remaining, preferred_order + remaining]

# optional: custom node colors
node_colors = {
    "Cancer": "#C44E52",
    "Epithelial": "#DD8452",
    "Fibroblast": "#55A868",
    "Myeloid": "#8172B3",
    "B cell": "#937860",
    "T cell": "#DA8BC3",
}

cmap = {n: node_colors.get(n, "#999999") for n in square.index}

circos = Circos.chord_diagram(
    square,
    space=4,
    cmap={n: node_colors.get(n, "#999999") for n in square.index},
    link_cmap=[(src, dst, node_colors.get(src, "#999999")) for src in square.index for dst in square.columns],
    label_kws=dict(size=12),
    link_kws=dict(alpha=0.65, lw=0.6, ec="white", direction=1, arrow_length_ratio=0.06),
)

fig = circos.plotfig(figsize=(4,4))
fig.suptitle(f"{factor} cell-cell communication program", y=1.02, fontsize=14)
fig.savefig(OUT_DIR + "PFI_factor7_chord.pdf", bbox_inches="tight")
plt.show()

In [ ]:
lr_cm = c2c.plotting.loading_clustermap(tensor_factorised.factors['Contexts'],
                                        loading_threshold=ccc_threshold, # To consider only top LRs
                                        use_zscore=True,
                                        figsize=(16, 9),
                                       )
plt.show()

In [ ]:
lr_cm = c2c.plotting.loading_clustermap(tensor_factorised.factors['Ligand-Receptor Pairs'],
                                        loading_threshold=0.08, # To consider only top LRs
                                        use_zscore=True,
                                        figsize=(16, 9),
                                       )
plt.show()

### Footprint enrichment analysis

In [ ]:
# load Progeny pathways
net = dc.op.progeny(organism='human', top=5000)

# re-load full list of ligand-receptor pairs
lr_pairs = li.resource.select_resource('consensus')

# generate ligand-receptor geneset
lr_progeny = li.rs.generate_lr_geneset(lr_pairs, net, lr_sep="^").rename(columns = {"interaction": "target"})
lr_progeny.head()

In [ ]:
lr_loadings_pfi = tensor_factorised.factors['Ligand-Receptor Pairs']

In [ ]:
# run enrichment analysis
estimate, pvals = dc.mt.ulm(lr_loadings_pfi.transpose(), lr_progeny, raw=False)

In [ ]:
from matplotlib.colors import TwoSlopeNorm

norm = TwoSlopeNorm(
    vcenter=0,
)
sns.clustermap(estimate, xticklabels=estimate.columns, cmap='coolwarm', norm=norm, z_score=4, figsize=(4,4))
plt.savefig(OUT_DIR + '/PROGENy.pdf', bbox_inches='tight')
plt.show()

In [ ]:
dc.pl.barplot(
    estimate,
    'Factor 1',
    vertical=False,
    cmap='coolwarm',
    dpi=150,
    figsize=(3,3),
    vmin=-7, vmax=7,
)
plt.savefig(OUT_DIR + '/Factor-5_progeny.pdf', bbox_inches='tight')
plt.show()

In [ ]:
dc.pl.barplot(
    estimate,
    'Factor 7',
    vertical=False,
    cmap='coolwarm',
    dpi=150,
    figsize=(3,3),
    vmin=-7, vmax=7,
)
plt.savefig(OUT_DIR + '/Factor-7_progeny.pdf', bbox_inches='tight')
plt.show()

In [ ]:
lr_progeny = li.rs.generate_lr_geneset(lr_pairs, net, lr_sep="^")
lr_progeny.head()

In [ ]:
selected_factor = "Factor 1"
pathway = 'TGFb'
# loadings to long format
lr_loadings_long = lr_loadings_pfi.reset_index().melt(id_vars='index', var_name="Factor", value_name="Loadings").rename(columns={'index':'interaction'})
# join progeny weights and keep only pathway
lr_loadings_long = lr_loadings_long.merge(lr_progeny, on='interaction').query("source == '{}' and Factor == '{}'".format(pathway, selected_factor))
# add sign to the weights
lr_loadings_long['sign'] = lr_loadings_long['weight'].apply(lambda x: 'positive' if x > 0 else 'negative')
# keep only relevant interactions for labels
lr_loadings_long['relevant_interactions'] = lr_loadings_long.apply(lambda x: x['interaction'] if (x['Loadings'] > 0.025) and (x['weight'] > 2.5) else None, axis=1)
lr_loadings_long.head()

In [ ]:
plot = (p9.ggplot(lr_loadings_long,
           p9.aes(x='weight', y='Loadings')) +
 p9.geom_smooth(method='lm') +
 p9.geom_point(p9.aes(colour='sign')) +
 p9.theme_bw() +
 p9.theme(legend_position='none') +
 p9.labs(title="{} | {}".format(pathway, selected_factor), x="PROGENy Weights", y="Loadings") +
 p9.scale_colour_manual(values=["royalblue", "red"]) +
 p9.geom_label(p9.aes(label='relevant_interactions'), size=9, nudge_y=0.02, nudge_x=0.0) +
 p9.xlim(1.5, 5)
 )
plot.save(OUT_DIR + '/Factor-5_TGFb_loadings.pdf')
plot.show()

### GSEA

In [ ]:
organism = 'human'
pathwaydb = 'GOBP' # Pathways

lr_list = ['^'.join(row) for idx, row in lr_pairs.iterrows()]
lr_set = c2c.external.generate_lr_geneset(lr_list,
                                          complex_sep='_',
                                          lr_sep='^',
                                          organism=organism,
                                          pathwaydb=pathwaydb,
                                          readable_name=True,
                                          output_folder=OUT_DIR
                                          )

In [ ]:
lr_loadings_pfi

In [ ]:
%%time
pvals, scores, gsea_df = c2c.external.run_gsea(loadings=lr_loadings_pfi,
                                               lr_set=lr_set,
                                               output_folder=OUT_DIR,
                                               # weight=1,
                                               min_size=15,
                                               permutations=999,
                                               processes=6,
                                               random_state=6,
                                               significance_threshold=0.05,
                                              )

In [ ]:
gsea_df.sort_values('Adj. P-value')